# מעבדה 03 — הסתברות והתהוות

הולך אחד אינו מספר לכם דבר. עשרת אלפים מספרים לכם הכול.

במעבדה הזו תצפו בעקומת פעמון בת-שחזור מרכיבה את עצמה מתוך הולכים בלתי צפויים בנפרד, תמדדו את
רוחבה ואת צורתה מול תאוריה שחוזה את שניהם — ואז תשברו אותה במכוון, כדי לגלות איזו הנחה החזיקה
אותה.

עבדו לפי הסדר. במקומות שבהם המחברת מבקשת מכם ניבוי, רשמו את הניבוי בתא המיועד **לפני** הרצת
התא הבא. זה אינו טקס: ניבוי שהתחייבתם אליו הוא הדרך האמינה היחידה לגלות שטעיתם.

## מפרט המודל

| | |
|---|---|
| **מערכת** | $N$ הולכים בלתי תלויים על ישר בלתי חסום; כל מיקום הוא הסכום המצטבר של צעדיו של אותו הולך |
| **דינמיקה** | בכל פעימה כל הולך מוסיף דגימה טרייה אחת מהתפלגות הצעדים |
| **גבול** | אין — הישר בלתי חסום; $N$, מספר הצעדים והתפלגות הצעדים קבועים בכל הרצה |
| **צבר** | ניסויים בלתי תלויים; כל הולך חוזר על אותו ניסוי ודבר אינו מקיים אינטראקציה |
| **מוזנח** | כל מה שפיזיקלי — אין מדיום, אין התנגשויות, אין אנרגיה, אין יחידות |
| **תקף כאשר** | הצעדים אכן בלתי תלויים ובעלי שונות סופית |
| **אופני כישלון** | צעדים מתואמים, צעדים בעלי שונות אינסופית, כל שאלה על מסלולו של הולך בודד מעבר להתפלגותו |

כל קוד המודל נמצא ב-`thermolab.sampling` — פתחו אותו וקראו. שום דבר בקורס הזה אינו מוסתר בתוך
מסגרת עבודה.

In [ ]:
# JupyterLite runs this notebook in the browser, where the course package and a few pure-
# Python libraries have to be installed into the kernel first. Under a local Jupyter they
# are already importable and this whole cell does nothing.
#
# thermolab is installed without its dependency graph on purpose: Pyodide supplies its own
# builds of numpy, scipy, matplotlib and sympy, older than the versions resolved for the
# development environment, and asking for those floors would send the installer to PyPI for
# packages that have no WebAssembly wheels. Add any new *pure-Python* dependency of
# thermolab to the list below.
try:
    import piplite
except ImportError:
    pass
else:
    await piplite.install(["pint", "ipywidgets", "jupyterquiz"])
    await piplite.install("thermolab", deps=False)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from thermolab import sampling
from thermolab.validation import relative_error, scaling_exponent, seed_study

# Every stochastic function takes its generator explicitly, so results are reproducible and
# no hidden global state can leak between cells.
rng = np.random.default_rng(2024)

N_STEPS = 1024
N_WALKERS = 10_000

print("step distributions, with the exact moments the overlays are built from:")
for name in ("pm1", "biased", "uniform", "heavy"):
    step = sampling.step_distribution(name)
    print(f"  {name:8s} mu_1 = {step.mean:+.4f}   sigma_1 = {step.std:.4f}")

## חלק 1 — הולך אחד אינו מנבא דבר

שישה הולכים, אותם כללים בדיוק, אותו מספר צעדים. הביטו היכן הם מסיימים.

In [ ]:
paths = sampling.random_walk(6, N_STEPS, rng)

plt.figure(figsize=(9, 3.6))
for index, path in enumerate(paths):
    plt.plot(path, lw=1.0, label=f"walker {index + 1}")
plt.axhline(0.0, color="0.6", ls=":")
plt.xlabel("steps")
plt.ylabel("position")
plt.title("six walkers, identical rules, nothing in common")
plt.legend(ncol=6, fontsize=8)
plt.tight_layout()
plt.show()

print("final positions:", paths[:, -1].astype(int))
print("typical distance predicted by sigma_1 sqrt(t):", round(float(np.sqrt(N_STEPS)), 1))

### נבאו

לפני הרצת התא הבא, רשמו כיצד תיראה ההיסטוגרמה של **עשרת אלפים** מיקומים סופיים. היו ספציפיים
לגבי שלושה דברים: היכן היא תהיה ממורכזת, מה יהיה רוחבה בקירוב, והאם אתם מצפים לאותה תשובה
בהרצה חוזרת עם זרע אחר.

**הניבוי שלכם:**

*(כתבו כאן לפני הרצת התא הבא)*

In [ ]:
trajectories = sampling.random_walk(N_WALKERS, N_STEPS, rng)

plt.figure(figsize=(8.5, 4.4))
for t in (16, 64, 256, 1024):
    # lattice=2.0 because a +/-1 walk can only stand on sites two apart. Bins that ignore
    # that cover different numbers of reachable sites and the histogram alternates
    # tall/short by tens of percent with nothing physical behind it.
    centres, density = sampling.walker_histogram(trajectories[:, t], n_bins=45, lattice=2.0)
    (line,) = plt.plot(centres, density, lw=1.4, label=f"t = {t}")
    # Dashed: the Gaussian predicted from the step distribution and t alone. Nothing here is
    # fitted to the histogram it is drawn on top of.
    plt.plot(centres, sampling.gaussian_limit(centres, t), color=line.get_color(), ls="--", lw=1.0)
plt.xlabel("position")
plt.ylabel("density")
plt.title("solid: measured cloud.   dashed: predicted Gaussian, not fitted")
plt.legend()
plt.tight_layout()
plt.show()

## חלק 2 — הרוחב: $\sigma$ גדל כמו $\sqrt{t}$

הענן מתפשט. מדדו באיזו מהירות, והשוו עם $\sigma_1\sqrt{t}$ — התוצאה הנובעת מלא יותר מאשר
"תוחלות נסכמות, ושונויות נסכמות עבור גדלים בלתי תלויים".

In [ ]:
spread = sampling.walker_spread(trajectories)
times = [4, 16, 64, 256, 1024]
measured = [float(spread[t]) for t in times]
exponent = scaling_exponent(times, measured)

for t, sigma in zip(times, measured, strict=True):
    print(f"t = {t:5d}   sigma = {sigma:7.3f}   sqrt(t) = {np.sqrt(t):7.3f}")
print(f"\nfitted exponent = {exponent:.4f}   (theory: +0.500)")

plt.figure(figsize=(6, 4.2))
plt.loglog(times, measured, "o", label="measured")
plt.loglog(times, np.sqrt(times), "-", label=r"$\sigma_1\sqrt{t}$")
plt.xlabel("t")
plt.ylabel(r"$\sigma(t)$")
plt.title(f"cloud width (fitted slope {exponent:.3f})")
plt.legend()
plt.tight_layout()
plt.show()

כעת השאלה שבה חיה התפיסה השגויה השנייה של המודול. הענן ללא ספק מתרחב. האם ה**ממוצע** שלו הולך
לאנשהו?

הרצועה האפורה שלהלן היא אי-הוודאות של התוחלת עצמה, $\sigma_1\sqrt{t}/\sqrt{N}$ — שגדלה אף היא.
תוחלת נמדדת יכולה להיקרא אפס רק ביחס לפס השגיאה שלה עצמה.

In [ ]:
mean = trajectories.mean(axis=0)
standard_error = np.sqrt(np.arange(N_STEPS + 1) / N_WALKERS)

plt.figure(figsize=(8.5, 4.0))
plt.plot(spread, lw=1.6, label=r"spread $\sigma(t)$")
plt.plot(mean, lw=1.4, color="0.2", label=r"mean $\langle x_t \rangle$")
plt.fill_between(np.arange(N_STEPS + 1), -3 * standard_error, 3 * standard_error,
                 color="0.7", alpha=0.5, lw=0, label="3 standard errors on the mean")
plt.axhline(0.0, color="0.6", ls=":")
plt.xlabel("steps")
plt.ylabel("position")
plt.legend()
plt.tight_layout()
plt.show()

print(f"mean at t = {N_STEPS}: {mean[-1]:+.3f}  (3 SE = {3 * standard_error[-1]:.3f})")
print(f"spread at t = {N_STEPS}: {spread[-1]:.3f}")

המעריך אוניברסלי. ה**מקדם** אינו: הוא $\sigma_1$, הפיזור של צעד בודד, והוא הדבר היחיד בהתפלגות
הצעדים ששורד אל התשובה.

In [ ]:
print(f"{'step':8s} {'sigma(400) measured':>20s} {'sigma_1 sqrt(400)':>18s} {'rel. error':>11s}")
for name in ("pm1", "biased", "uniform", "heavy"):
    step = sampling.step_distribution(name)
    walk = sampling.random_walk(4000, 400, rng, step=step)
    sigma = float(sampling.walker_spread(walk)[-1])
    predicted = step.std * np.sqrt(400)
    print(f"{name:8s} {sigma:20.3f} {predicted:18.3f} {relative_error(sigma, predicted):11.3f}")

### המדידה

כל מספר בקורס הזה שיש לו חשיבות מדווח בצורה `value ± error`, והשגיאה הזו היא **שגיאת תקן**:
הפיזור של תוחלת מוערכת, ולא הפיזור של הנתונים. כאן ה"נתונים" הם שמונה חזרות בלתי תלויות של
הניסוי כולו, כל אחת עם הזרע שלה.

In [ ]:
def fitted_exponent(generator):
    """Repeat the entire Part 2 measurement under one fresh generator."""
    walk = sampling.random_walk(3000, 1024, generator)
    walk_spread = sampling.walker_spread(walk)
    return scaling_exponent(times, [float(walk_spread[t]) for t in times])


study = seed_study(fitted_exponent, n_seeds=8)

print(f"seed-to-seed values: {np.round(study.values, 4)}")
print(f"\nmeasurement:  alpha = {study.mean:.4f} +/- {study.standard_error:.4f}   (target 0.500)")
print(f"agrees with 0.5 at 3 sigma: {study.agrees_with(0.5, n_sigma=3.0)}")

## חלק 3 — הצורה: משפט הגבול המרכזי

הרוחב הגיע משתי שורות של אלגברה. הצורה היא משפט, וטענתו חזקה בהרבה ממה שהיא נשמעת בתחילה:
תקננו את סכום $n$ הצעדים,

$$
Z_n = \frac{S_n - n\mu_1}{\sigma_1\sqrt{n}} ,
$$

וככל ש-$n$ גדל התפלגותו של $Z_n$ מתקרבת אל *אותה* עקומה, לא משנה כיצד נראו הצעדים. בכל פאנל
שלהלן משורטטות שלוש התפלגויות צעדים שאין ביניהן דבר משותף.

In [ ]:
grid = np.linspace(-4.0, 4.0, 200)
standard_gaussian = np.exp(-0.5 * grid**2) / np.sqrt(2.0 * np.pi)

fig, axes = plt.subplots(1, 4, figsize=(13, 3.3), sharey=True)
for ax, n_terms in zip(axes, (1, 4, 16, 256), strict=True):
    for name in ("pm1", "uniform", "heavy"):
        standardized = sampling.clt_sum_distribution(n_terms, 20_000, rng, step=name)
        # Coin sums land on a lattice of spacing 2/sqrt(n); the other two step distributions
        # are continuous and need no alignment. Leave the coin unaligned and it alternates
        # tall/short — which reads as the one distribution refusing to collapse, when in fact
        # it is the one whose bins were drawn wrong.
        centres, density = sampling.walker_histogram(
            standardized,
            n_bins=49,
            span=(-4.0, 4.0),
            lattice=2.0 / np.sqrt(n_terms) if name == "pm1" else None,
        )
        ax.plot(centres, density, lw=1.3, label=name)
    ax.plot(grid, standard_gaussian, color="crimson", ls="--", lw=1.3)
    ax.set_title(f"n = {n_terms}")
    ax.set_xlabel(r"$Z_n$")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## חלק 4 — האם ההילוך נמשך בחזרה?

כמעט כולם מאמינים, ברמה כלשהי, שהולך שהתרחק הרחק ימינה עתיד איכשהו לחזור. המודל אינו מכיל שום
מנגנון כזה — אך במקום להתווכח על כך, התנו על ההולכים ש*אכן* נמצאים הרחק ימינה ומדדו מה הם עושים
מכאן ואילך.

In [ ]:
threshold = 30.0
far_right = trajectories[:, 500] >= threshold

later = trajectories[far_right, 1000] - trajectories[far_right, 500]
everyone = trajectories[:, 1000] - trajectories[:, 500]

print(f"{far_right.sum()} of {N_WALKERS} walkers stood at x >= +{threshold:.0f} after 500 steps.")
print("\nMean displacement over their NEXT 500 steps")
print(f"  those walkers : {later.mean():+.3f} +/- {later.std(ddof=1) / np.sqrt(later.size):.3f}")
print(f"  everyone      : {everyone.mean():+.3f} "
      f"+/- {everyone.std(ddof=1) / np.sqrt(everyone.size):.3f}")
print(f"\nA restoring force would have to predict about {-threshold:+.0f}.")

plt.figure(figsize=(7, 3.6))
plt.hist(everyone, bins=60, density=True, alpha=0.45, label="all walkers")
plt.hist(later, bins=60, density=True, histtype="step", lw=1.6, label="only those at x >= +30")
plt.axvline(0.0, color="0.4", ls=":")
plt.xlabel("displacement over steps 500-1000")
plt.ylabel("density")
plt.legend()
plt.tight_layout()
plt.show()

## חלק 5 — לשבור אותה במכוון

כל מה שקדם נשען על הנחה אחת: הצעדים **בלתי תלויים**. תנו להילוך *התמדה* במקום זאת — בכל צעד
הוא חוזר על צעדו הקודם בהסתברות $q$, ואחרת דוגם מחדש. הצעדים עדיין בעלי אותה התפלגות, עדיין
$\pm 1$, עדיין בעלי שונות סופית. רק אי-התלות נעלמה.

תקננו באותו $\sigma_1\sqrt{n}$ בדיוק וראו מה שורד.

In [ ]:
n = 512
grid_wide = np.linspace(-6.0, 6.0, 240)
wide_gaussian = np.exp(-0.5 * grid_wide**2) / np.sqrt(2.0 * np.pi)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5), sharey=True)
for ax, q in zip(axes, (0.0, 0.5, 0.95), strict=True):
    walk = sampling.correlated_walk(6000, n, q, rng)
    final = walk[:, -1]

    naive = final / np.sqrt(n)
    centres, density = sampling.walker_histogram(
        naive, n_bins=49, span=(-6.0, 6.0), lattice=2.0 / np.sqrt(n)
    )
    ax.plot(centres, density, lw=1.5, label=r"standardized by $\sigma_1\sqrt{n}$")
    ax.plot(grid_wide, wide_gaussian, color="crimson", ls="--", lw=1.3, label=r"$N(0,1)$")
    ax.set_title(f"q = {q}")
    ax.set_xlabel(r"$Z_n$")

    factor = float(final.var(ddof=1) / n)
    print(f"q = {q:4.2f}   Var(x_t)/t measured {factor:7.3f}   predicted (1+q)/(1-q) = "
          f"{(1 + q) / (1 - q):7.3f}")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

אז מה בדיוק נשבר? לא צורת הפעמון — ההילוך המתמיד עדיין גאוסי בזמנים ארוכים. מה שנשבר הוא הצעד
שהופך נתוני צעד בודד לרוחב, משום שהשונויות מפסיקות להיסכם פשוט ברגע שהצעדים מתואמים.

תקננו את ההילוך עם $q = 0.95$ לפי הפיזור ה*אמיתי* שלו במקום זאת, וצפו בו נופל לתלם.

In [ ]:
q = 0.95
walk = sampling.correlated_walk(6000, n, q, rng)
final = walk[:, -1]

inflation = (1.0 + q) / (1.0 - q)
naive = final / np.sqrt(n)
repaired = final / np.sqrt(n * inflation)

plt.figure(figsize=(7.5, 4.0))
# Dividing by a different number rescales the lattice by the same factor, so each curve gets
# its own spacing -- 2/sqrt(n) for the naive standardization, and that shrunk by sqrt of the
# inflation factor for the repaired one.
for values, lattice, label, style in (
    (naive, 2.0 / np.sqrt(n), r"divided by $\sigma_1\sqrt{n}$", "-"),
    (repaired, 2.0 / np.sqrt(n * inflation), r"divided by the true spread", "-."),
):
    centres, density = sampling.walker_histogram(
        values, n_bins=61, span=(-8.0, 8.0), lattice=lattice
    )
    plt.plot(centres, density, style, lw=1.6, label=label)
grid_repair = np.linspace(-8.0, 8.0, 320)
plt.plot(grid_repair, np.exp(-0.5 * grid_repair**2) / np.sqrt(2.0 * np.pi),
         color="crimson", ls="--", lw=1.3, label=r"$N(0,1)$")
plt.xlabel(r"$Z_n$")
plt.ylabel("density")
plt.title(f"persistent walk, q = {q}: the shape was never the problem")
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

## חלק 6 — לוח גלטון: דה-מואבר–לפלס בעץ

חרוז הנופל דרך מערך משולש של יתדות פונה ימינה או שמאלה בכל אחד מהם. לוח בן 12 שורות הוא אפוא
הילוך מטבע בן 12 צעדים, והערימה בתחתית היא התפלגות בינומית — נמדדת בעץ ובכבידה ולא במחולל
מספרים אקראיים.

הקובץ `data/galton-board.csv` מכיל הרצה אחת של 200 חרוזים (קראו את כותרת הקובץ כדי לדעת בדיוק
מה הוא כן ומה אינו). אם יש ברשותכם לוח פיזי, או אם הרצתם את פרוטוקול הטלות המטבע הכיתתי,
החליפו את הקובץ בספירות שלכם והריצו את התא הזה מחדש — זהו תרגיל טוב יותר עם הנתונים שלכם.

In [ ]:
from pathlib import Path

try:
    import piplite  # noqa: F401
except ImportError:
    # Desktop / nbmake: the repository root is three directories up from this notebook.
    csv_path = Path("..", "..", "..", "data", "galton-board.csv")
else:
    # JupyterLite bundles only the notebooks/ tree (jupyter_lite_config.json's
    # LiteBuildConfig.contents), so the browser gets its own copy of the CSV co-located here.
    csv_path = Path("data", "galton-board.csv")

board = np.loadtxt(csv_path, delimiter=",", comments="#")
bin_index, counts = board[:, 0], board[:, 1]
n_rows, n_beads = int(bin_index.max()), counts.sum()

k, pmf, gaussian = sampling.binomial_to_gaussian(n_rows, 0.5)

plt.figure(figsize=(7.5, 4.0))
plt.bar(bin_index, counts, width=0.8, alpha=0.45, label=f"{int(n_beads):d} beads")
plt.plot(k, pmf * n_beads, "o-", lw=1.4, label="exact binomial")
plt.plot(k, gaussian * n_beads, "--", color="crimson", lw=1.4, label="de Moivre-Laplace")
plt.xlabel("bin (number of right-turns)")
plt.ylabel("beads")
plt.title(f"a {n_rows}-row Galton board is a {n_rows}-step coin walk")
plt.legend()
plt.tight_layout()
plt.show()

print(f"measured mean bin  {float((bin_index * counts).sum() / n_beads):.3f}   "
      f"binomial np = {n_rows * 0.5:.3f}")
print(f"peak: exact {float(pmf[n_rows // 2]):.4f}   Gaussian {float(gaussian[n_rows // 2]):.4f}")
print(f"\nWith only {int(n_beads):d} beads the standard error on each bin count is of order "
      f"{np.sqrt(counts.max()):.1f},")
print("which is why the bars and the curve are allowed to disagree by as much as they do.")

## חלק 7 — בדיקות אוטומטיות

סימולציה שלא בדקתם היא תמונה, לא ראיה. אלה אותן טענות בדיוק הרצות במערך המבחנים של הפרויקט.

In [ ]:
# 1. Probability is conserved: the binomial pmf sums to 1 even at n = 200, where the
#    coefficients themselves are astronomically large.
_, pmf_check, _ = sampling.binomial_to_gaussian(200, 0.5)
assert abs(float(pmf_check.sum()) - 1.0) < 1e-12

# 2. Additivity, for a step distribution that is nothing like a coin.
heavy = sampling.step_distribution("heavy")
check = sampling.random_walk(8000, 512, np.random.default_rng(5), step=heavy)
assert relative_error(float(check[:, -1].var(ddof=1)), heavy.variance * 512) < 0.1

# 3. de Moivre-Laplace at the peak: 1/sqrt(50 pi) against the exact binomial.
_, pmf_100, gaussian_100 = sampling.binomial_to_gaussian(100, 0.5)
assert relative_error(float(gaussian_100[50]), float(pmf_100[50])) < 0.005
assert abs(float(gaussian_100[50]) - 1.0 / np.sqrt(50.0 * np.pi)) < 1e-12

# 4. The exponent, across independent seeds, with an honest error bar.
assert study.agrees_with(0.5, n_sigma=3.0)

# 5. The counterexample really is one: correlated steps inflate the variance by (1+q)/(1-q).
broken = sampling.correlated_walk(6000, 512, 0.9, np.random.default_rng(6))
assert relative_error(float(broken[:, -1].var(ddof=1) / 512), 19.0) < 0.2

print(f"binomial pmf sum      {float(pmf_check.sum()):.12f}")
print(f"heavy-step Var/t      {float(check[:, -1].var(ddof=1)) / 512:.4f}  "
      f"(sigma_1^2 = {heavy.variance:.4f})")
print(f"peak Gaussian/exact   {float(gaussian_100[50]) / float(pmf_100[50]):.6f}")
print(f"exponent              {study.mean:.4f} +/- {study.standard_error:.4f}")
print(f"correlated Var/t      {float(broken[:, -1].var(ddof=1)) / 512:.3f}  (predicted 19.000)")
print("\nall checks passed")

## חלק 8 — חקרו בעצמכם

קבעו את המחוונים ואז לחצו על **Run Interact**. (הפונקציה משרטטת מחדש שני פאנלים, ולכן היא רצה
לפי דרישה ולא בכל תזוזה של מחוון.)

שלושה ניסויים שכדאי לעשות:

1. השאירו הכול קבוע ושנו רק את התפלגות הצעדים. צפו ברוחב משתנה ובצורה שאינה משתנה.
2. העלו את מחוון ההתמדה בהדרגה. מצאו את הערך שבו הפיזור הנמדד עוזב באופן נראה לעין את הניבוי
   המקווקו — ושימו לב כמה מתאם דרוש לשם כך.
3. הורידו את מספר ההולכים ל-50. הענן עדיין שם, אך האם משהו בו בר-שחזור? זהו החוק $N^{-1/2}$
   ממודול 0, נראה מן הקצה השני.

In [ ]:
import ipywidgets as widgets


def explore(n_walkers=2000, n_steps=500, step="pm1", persistence=0.0):
    generator = np.random.default_rng(0)
    if persistence > 0.0:
        walk = sampling.correlated_walk(n_walkers, n_steps, persistence, generator)
        described = f"persistent +/-1 steps, q = {persistence:.2f}"
    else:
        walk = sampling.random_walk(n_walkers, n_steps, generator, step=step)
        described = f"independent steps, {step}"

    # The overlay always uses the INDEPENDENT-step prediction. That is the point: when the
    # persistence slider is up, the gap between curve and histogram is the broken hypothesis.
    overlay_step = "pm1" if persistence > 0.0 else step
    walk_spread = sampling.walker_spread(walk)
    ticks = np.arange(n_steps + 1)

    fig, (left, right) = plt.subplots(1, 2, figsize=(11, 3.6))
    left.plot(ticks, walk_spread, lw=1.6, label="measured")
    left.plot(ticks, sampling.step_distribution(overlay_step).std * np.sqrt(ticks),
              color="crimson", ls="--", lw=1.3, label=r"$\sigma_1\sqrt{t}$")
    left.set_xlabel("steps")
    left.set_ylabel("spread")
    left.set_title(described)
    left.legend(fontsize=8)

    # The +/-1 families (and the persistent walk) sit on a lattice of spacing 2; the uniform
    # and heavy-tailed steps are continuous and take no alignment.
    lattice = 2.0 if persistence > 0.0 or step in ("pm1", "biased") else None
    centres, density = sampling.walker_histogram(walk[:, -1], n_bins=41, lattice=lattice)
    right.plot(centres, density, lw=1.5, label="measured")
    right.plot(centres, sampling.gaussian_limit(centres, n_steps, overlay_step),
               color="crimson", ls="--", lw=1.3, label="predicted")
    right.set_xlabel("position")
    right.set_ylabel("density")
    right.set_title(f"final positions, t = {n_steps}")
    right.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print(f"measured spread {walk_spread[-1]:.3f}   "
          f"independent-step prediction "
          f"{sampling.step_distribution(overlay_step).std * np.sqrt(n_steps):.3f}")


widgets.interact_manual(
    explore,
    n_walkers=widgets.IntSlider(min=50, max=8000, step=50, value=2000, description="walkers"),
    n_steps=widgets.IntSlider(min=10, max=2000, step=10, value=500, description="steps"),
    step=widgets.Dropdown(options=["pm1", "biased", "uniform", "heavy"], description="step"),
    persistence=widgets.FloatSlider(min=0.0, max=0.95, step=0.05, value=0.0, description="q"),
);

## בדקו את הבנתכם

הריצו את התא שלהלן לחידון עם בדיקה אוטומטית. אותן שאלות, עם הסברים כתובים לכל אפשרות, נמצאות
בדף המודול.

In [ ]:
import json
from pathlib import Path

quiz_path = Path("..") / "_quiz" / "03-random-walks.json"
if quiz_path.exists():
    from jupyterquiz import display_quiz

    # Parsed here with an explicit encoding: jupyterquiz opens the file with the platform
    # default, which cannot decode the Hebrew edition of this notebook on Windows.
    display_quiz(json.loads(quiz_path.read_text(encoding="utf-8")))
else:
    print("Quiz not generated yet — run: uv run python scripts/render_quizzes.py")

## לפני שאתם עוזבים

כתבו כמה משפטים על כל אחת מהשאלות, בתא שלהלן.

1. מה ניבאתם שהתברר כשגוי, ומה בדיוק היה הפגם בהיגיון שלכם?
2. חלק 5 שבר הנחה אחת של משפט הגבול המרכזי, וההיסטוגרמה עדיין יצאה גאוסית לאחר תקנון נכון.
   נסחו במדויק מה הכישלון של ההנחה השבורה עלה לכם ומה הוא לא.
3. עמית מדווח על מדידה בצורה `4.71 ± 0.03` מתוך 900 חזרות. מהו ה-0.03, מה הוא היה מתוך 100
   חזרות, ואיזה מספר בנתונים שלו *אינו* מושפע ממספר החזרות שהוא הריץ?

**התשובות שלכם:**

1.
2.
3.